Complete protein preparation.
1. dowload pdb file
2. check how many chains are there. and retain the one we want.
3. Check how many heteroatoms are there and retain what we want.
4. Use pdbfixer to add missing residues.
5. Then use OpenMM Modeller to add hydrogens.
6. Visualize using py3Dmol

!openMM-setup

In [9]:
import requests
from Bio.PDB import PDBParser, PDBIO, Select
from rdkit import Chem


In [10]:
pdb_id = "7CMD"   # Replace with your PDB ID

url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

response = requests.get(url)

with open(f"{pdb_id}.pdb", "wb") as f:
    f.write(response.content)

print(f"{pdb_id}.pdb downloaded.")

7CMD.pdb downloaded.


======================pdbfixer========================

In [11]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename="7CMD.pdb")

# Check chain order and IDs
for i, chain in enumerate(fixer.topology.chains()):
    print(i, chain.id)

0 A
1 B
2 C
3 D
4 A
5 B
6 C
7 D
8 A
9 B
10 C
11 D


In [12]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

# Keep only chain A
chains_to_remove = [
    i for i, chain in enumerate(fixer.topology.chains())
    if chain.id != "A"
]

fixer.removeChains(chains_to_remove)

In [13]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer.findMissingResidues()
print("Missing residues:", fixer.missingResidues)

fixer.findNonstandardResidues()
print("Nonstandard residues:", fixer.nonstandardResidues)



Missing residues: {(0, 221): ['GLN', 'ILE', 'PRO', 'CYS', 'THR', 'CYS', 'GLY', 'LYS', 'GLN', 'ALA'], (0, 306): ['PRO', 'VAL', 'THR']}
Nonstandard residues: []


###ADD MISSING RESIDUES####

In [14]:
fixer.replaceNonstandardResidues()
#fixer.removeHeterogens(False) #true will retain water. False deletes everything
fixer.findMissingAtoms()
fixer.addMissingAtoms()

fixer.findMissingResidues()
print("Missing residues:", fixer.missingResidues)

fixer.findNonstandardResidues()
print("Nonstandard residues:", fixer.nonstandardResidues)

fixer.addMissingHydrogens(7.4)
#PDBFile.writeFile(fixer.topology, fixer.positions, open('output_pdbfixer.pdb', 'w'))

Missing residues: {}
Nonstandard residues: []


In [17]:
#I want to retain ZN#
from openmm.app import Modeller, PDBFile

modeller = Modeller(fixer.topology, fixer.positions)

standard = {
    "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY",
    "HIS","ILE","LEU","LYS","MET","PHE","PRO","SER",
    "THR","TRP","TYR","VAL",
    "ASH","GLH","HID","HIE","HIP","LYN","CYX"
}

delete = []

for residue in modeller.topology.residues():
    if residue.name not in standard and residue.name != "ZN":
        delete.append(residue)

modeller.delete(delete)

with open("output_pdbfixer.pdb", "w") as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)

In [18]:
import py3Dmol

with open("output_pdbfixer.pdb", "r") as f:
    pdb = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb, "pdb")

# Protein
#view.setStyle({"cartoon": {"color": "spectrum"}})
view.setStyle(
        {"cartoon": {"color": "spectrum"}}
)

# Ligand and other non-water hetero atoms
view.addStyle(
    {"hetflag": True, "not": {"resn": ["HOH", "WAT", "TIP3", "SOL"]}},
    {"stick": {}}
)

# Waters
view.addStyle(
    {"resn": ["HOH", "WAT", "TIP3", "SOL"]},
    {"sphere": {"radius": 0.18}}
)

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.